# Word Embeddings and Language Bias – Project Notebook

Use this notebook for carrying out the analyses from the workshop notebook on your own subreddit data.

NB. Make sure you've installed all required packages (code is in the Lesson file).

### Icons Used in This Notebook
💭 **Reflection**: Reflecting on ethical implications, biases, and social impact in data science.<br>
🔔 **Question**: A quick question to help you understand what's going on.<br>
💡 **Tip**: How to do something a bit more efficiently or effectively.<br>
⚠️ **Warning**: Heads-up about tricky stuff or common mistakes.

> **Data transparency note**: The analyses in this notebook operate on text written by real people in the community you collected. The patterns these methods surface — including the biases you will measure — reflect that community's discourse, not universal truths about language or the people writing. Keep this context in mind when interpreting results.

In [ ]:
# Package imports
import os
import pandas as pd
import numpy as np

import pickle
from gensim.models import Word2Vec
import multiprocessing

import matplotlib.pyplot as plt
from sklearn.manifold import TSNE

import nltk
nltk.download('averaged_perceptron_tagger_eng')

import bokeh
from bokeh.plotting import figure, show, output_notebook
from bokeh.models import HoverTool, ColumnDataSource, LabelSet

output_notebook()
bokeh.io.output_notebook()

## Loading the Data

Replace `YOUR_FILE_PP.csv` below with the name of the preprocessed file you saved in the week 1 project notebook. Word embeddings work best on **comments** data (more text, more varied contexts), but submissions work too.

In [ ]:
df = pd.read_csv('../../data/YOUR_FILE_PP.csv')

In [ ]:
df.head(3)

In [ ]:
# Remove all rows that are '[removed]' or '[deleted]'
df = df.loc[~df['pp_text'].isin(['[removed]', '[deleted]' ]),:]

# Select only rows that have >3 characters in selftext
df = df.loc[df['pp_text'].str.len() > 3]

In [ ]:
from tqdm import tqdm 

lemmas_split = [lemma.split() for lemma in tqdm(df['pp_text'])]

# Constructing a Word2Vec Model

In [ ]:
cores = min(4, multiprocessing.cpu_count()) # Number of cores to use

n_features = 300     # Word vector dimensionality (how many features each word will be given)
min_word_count = 10  # Minimum word count to be taken into account
n_workers = cores    # Number of threads to run in parallel
window = 5           # Context window size
downsampling = 1e-2  # Downsample setting for frequent words
seed = 1             # Seed for the random number generator (to create reproducible results)
sg = 1               # Skip-gram = 1, CBOW = 0
epochs = 20          # Number of iterations over the corpus

model = Word2Vec(
    sentences=lemmas_split,
    workers=n_workers,
    vector_size=n_features,
    min_count=min_word_count,
    window=window,
    sample=downsampling,
    seed=seed,
    sg=sg)

💡 **Tip**: Training Word2Vec on ~100k documents with 300 dimensions and 20 epochs typically takes **2–5 minutes** on a modern laptop. Larger corpora (>500k documents) can take 30+ minutes. If you need faster results during exploration, reduce `n_features` to 100 or `epochs` to 5 — you can always retrain with full settings once you're happy with the pipeline.

⚠️ **Warning**: If you are working with a small corpus (fewer than ~5,000 documents), consider using a **pre-trained model** instead (see the note at the bottom of this notebook). Training Word2Vec requires substantial data to produce meaningful embeddings; small datasets will often yield erratic, unreliable results.

In [ ]:
# Save the model to disk
model.save('../../data/embeddings.emb')

In [ ]:
# Load the model from disk
model = Word2Vec.load('../../data/embeddings.emb')

How many terms are in your vocabulary?

In [ ]:
len(model.wv)

# Word Similarity


In [ ]:
def get_most_similar_terms(model, token, topn=20):
    """Look up the top N most similar terms to the token."""
    for word, similarity in model.wv.most_similar(positive=[token], topn=topn):
        print(f"{word}: {round(similarity, 3)}")

Replace `CHOOSE_WORD` below with a word relevant to your community (make sure it appears in your corpus — check `model.wv.index_to_key[:50]` for frequent words if you're unsure).

In [ ]:
get_most_similar_terms(model, 'CHOOSE_WORD')

## Word Arithmetic

One famous property of well-trained word embeddings is that **vector arithmetic can capture relational meaning**. The classic example: `king - man + woman ≈ queen`. The model has learned that "king" and "queen" have the same relationship as "man" and "woman."

This works by supplying `positive` words (those to add) and `negative` words (those to subtract) to `most_similar()`. The result is the nearest term to the resulting vector.

These analogies can be powerful tools for exploring how your community constructs social categories — but also fragile ones. If the corpus doesn't contain enough examples to learn a relationship cleanly, the results may be noise.

🔔 **Question**: Try a few of your own analogies. What relationships does your community's discourse embed clearly? What relationships fail or produce surprising results?

In [ ]:
# Word arithmetic
# Syntax: most_similar(positive=['word_to_add'], negative=['word_to_subtract'])
# Replace WORD1/WORD2/WORD3 with words that appear in your corpus!

print("WORD1 + WORD2 - WORD3:")
for word, score in model.wv.most_similar(positive=['WORD1', 'WORD2'], negative=['WORD3'], topn=5):
    print(f"  {word}: {score:.3f}")

# Try more of your own:
# model.wv.most_similar(positive=['...', '...'], negative=['...'], topn=5)

# Visualizing High Dimensional Spaces with $t$-SNE

Change `words` to include words you are interested in for your data (make sure they appear in your dataset!) in order to visualize their relations. You can make this list as long or short as you want.

💭 **Reflection**: Figuring out **which words** you are interested in exploring is one of the main challenges when doing work like this! It will depends on your subreddit and your research questions.

In [ ]:
words = ['WORD1', 'WORD2', 'WORD3', 'WORD4','WORD5','WORD6','WORD7','WORD8']

# Extract the word vectors
word_vectors = np.array([model.wv[word] for word in words])

In [ ]:
# If you get an ImportError in the line tsne=TSNE(), you might need to install scikit-learn:
# %pip install -U scikit-learn 

In [ ]:
# Reduce dimensionality using t-SNE
tsne = TSNE(n_components=2, random_state=2, perplexity=2)
reduced_vectors = tsne.fit_transform(word_vectors)

In [ ]:
# Store the t-SNE vectors
words_df = pd.DataFrame(reduced_vectors,
                            index=pd.Index([word for word in words]),
                            columns=['x', 'y'])

In [ ]:
# Add our DataFrame as a ColumnDataSource for Bokeh
plot_data = ColumnDataSource(words_df)

# Create the plot and configure the title, dimensions, and tools
tsne_plot = figure(title='t-SNE Word Embeddings')

# Add a hover tool to display words on roll-over
tsne_plot.add_tools(HoverTool(tooltips='@index'))

# Draw the words as circles on the plot
tsne_plot.scatter('x', 'y',
                  source=plot_data,
                  color='blue',
                  size=10,
                  hover_line_color='black')

# Add labels to the points
labels = LabelSet(x='x', y='y', text='index', level='glyph',
                  x_offset=5, y_offset=5, source=plot_data)
tsne_plot.add_layout(labels)

# Engage!
show(tsne_plot)

In [ ]:
from bokeh.plotting import output_file, save

output_file("outputs_project/selected_word_embeddings_tsne.html")
save(tsne_plot)

Now let's use $t$-SNE to take **all** the word vectors.

⚠️ **Warning**: Running t-SNE on the full vocabulary can take several minutes (or longer for large vocabularies). The results are saved to disk below, so you only need to run this once.

In [ ]:
tsne = TSNE(init='pca', learning_rate='auto')
X_tsne = tsne.fit_transform(model.wv.vectors)

In [ ]:
# Store the t-SNE vectors
tsne_df = pd.DataFrame(X_tsne,
                            index=pd.Index(model.wv.index_to_key),
                            columns=['x', 'y'])

In [ ]:
# Create some filepaths to save our model
tsne_path = '../../data/tsne_model'
tsne_df_path = '../../data/tsne_df.pkl'

In [ ]:
# Save to disk
with open(tsne_path, 'wb') as f:
    pickle.dump(X_tsne, f)

tsne_df.to_pickle(tsne_df_path)

Here's a convenient code block to load this data, to start from this point:

In [ ]:
with open(tsne_path, 'rb') as f:
    X_tsne = pickle.load(f)
    
tsne_df = pd.read_pickle(tsne_df_path)

Visualize with `bokeh`.

In [ ]:
# Add our DataFrame as a ColumnDataSource for Bokeh
plot_data = ColumnDataSource(tsne_df)

# Create the plot and configure the title, dimensions, and tools
tsne_plot = figure(title='t-SNE Word Embeddings')

# Add a hover tool to display words on roll-over
tsne_plot.add_tools(HoverTool(tooltips='@index') )

# Draw the words as circles on the plot
tsne_plot.scatter('x', 'y',
                  source=plot_data,
                  color='blue',
                  line_alpha=0.2,
                  fill_alpha=0.1,
                  size=10,
                  hover_line_color='black')

# Engage!
show(tsne_plot)

In [ ]:
output_file("outputs_project/full_word_embeddings_tsne.html")
save(tsne_plot)

# Language Biases and Word Embeddings


In [ ]:
# Import function to calculate biased words
from utils import calculate_biased_words

💭 **Reflection**: You will have to change the following words to words that are illustrative of a **target concept**, organized in some kind of binary. Think of "male" and "female", "Islam" and "Christianity", or "career" and "family". For some examples of target sets that have been used in the literature on language biases, check the [bottom of this notebook](#targets).

In [ ]:
target1 = ['WORD1' , 'WORD2' , 'WORD3' , 'WORD4' , 'WORD5' , 'WORD6' , 'WORD7' , 'WORD8']
target2 = ['WORD1' , 'WORD2' , 'WORD3' , 'WORD4' , 'WORD5' , 'WORD6' , 'WORD7' , 'WORD8']

In [ ]:
model = Word2Vec.load('../../data/embeddings.emb')

In [ ]:
[b1, b2] = calculate_biased_words(model, target1, target2, 4)

Let's print some biases.

In [ ]:
print('Biased words towards target set 1')
print([word for word in b1.keys()])

In [ ]:
print('Biased words towards target set 2')
print([word for word in b2.keys()] )

## Visualizing Biases using $t$-SNE

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sklearn.manifold import TSNE
%matplotlib inline

In [ ]:
with open(tsne_path, 'rb') as f:
    X_tsne = pickle.load(f)
    
tsne_df = pd.read_pickle(tsne_df_path)

In [ ]:
# Convert biased term keys to arrays
target1_idx = np.array([model.wv.key_to_index[key] for key in b1.keys()])
target2_idx = np.array([model.wv.key_to_index[key] for key in b2.keys()])

In [ ]:
# Find t-sne values for the biased sets
X_target1 = X_tsne[target1_idx]
X_target2 = X_tsne[target2_idx]

In [ ]:
from bokeh.io import show, output_notebook, output_file
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource, LabelSet

# Save as HTML
output_file("outputs_project/biases_plot.html")  

# Set up the Bokeh plot
output_notebook()

p = figure()

# Create ColumnDataSource for X_target1 (blue)
source1 = ColumnDataSource(data=dict(x=X_target1[:, 0], y=X_target1[:, 1], label=[model.wv.index_to_key[idx] for idx in target1_idx]))

# Create ColumnDataSource for X_target2 (red)
source2 = ColumnDataSource(data=dict(x=X_target2[:, 0], y=X_target2[:, 1], label=[model.wv.index_to_key[idx] for idx in target2_idx]))

# Add scatter plot for X_target1 (blue)
p.scatter(x='x', y='y', color='blue', size=8, source=source1)

# Add scatter plot for X_target2 (red)
p.scatter(x='x', y='y', color='red', size=8, source=source2)

# Add labels for X_target1
labels1 = LabelSet(x='x', y='y', text='label', x_offset=6, y_offset=3, source=source1)
p.add_layout(labels1)

# Add labels for X_target2
labels2 = LabelSet(x='x', y='y', text='label', x_offset=6, y_offset=3, source=source2)
p.add_layout(labels2)

# Show the plot
show(p)

## 💭 Reflection: Bias as Discourse

Note that binary target concepts are often products of ideology and normativity in society. The gender binary is a prime example: by defining two target sets (male/female), we are already importing a binary structure that many people's lives exceed or complicate.

But there's a deeper question here. The bias we measure is not just "bias in the data" — it is **the community's discourse about itself**. When members of your community consistently use certain words near one concept vs another, they are enacting a set of social expectations, judgments, and relational norms. Word embeddings make these patterns visible at scale.

This is why word embeddings are a genuinely humanistic tool, not just a machine learning technique:
- They let you study **how a community talks**, not just what it talks about.
- They surface **implicit assumptions** that no individual poster may have consciously intended.
- They are **constructed from training data** — which means they reflect the past, and may not represent all members of the community equally.

🔔 **Question**: Look at the words biased toward each of your target sets. What do these patterns suggest about how your community thinks about these concepts? Does this match what you would expect?

⚠️ **Warning**: The biases you see here are specific to your corpus and its time period. They should not be read as claims about language in general — only about how this particular community spoke during the time the data was collected.

Also note that determining your own target concepts is an **iterative** process. Try changing some of the words in the target concepts and discuss with your classmates what makes for a coherent and robust target set.

## Note: Using Pre-trained Word2Vec Models

If your own corpus is too small to train meaningful embeddings (fewer than ~5,000 documents), or if you want to explore embeddings trained on a much larger and more general corpus, you can use **pre-trained models**.

The most commonly used pre-trained model is **Google's Word2Vec** trained on 100 billion words of Google News. It contains 3 million unique words, each represented by a 300-dimensional vector.

To use it:

```python
from gensim.models import KeyedVectors

# Download from: https://code.google.com/archive/p/word2vec/ (about 1.5GB)
# Then load:
pretrained = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin', binary=True)
pretrained.most_similar('community', topn=10)
```

**Trade-offs**:
| | Custom-trained (this notebook) | Pre-trained (Google News) |
|---|---|---|
| Domain relevance | High — trained on your community's text | Low — general news text |
| Vocabulary coverage | Limited to your corpus | 3 million words |
| Training cost | Minutes on your laptop | Already done |
| Research validity | Reflects your community's discourse | Reflects general English |

For the discourse analysis goals of this course, a community-specific model is usually more appropriate, even if it's noisier. A pre-trained model is best used for comparison or as a baseline.

<a id='targets'></a>
# Existing Target Sets

Here are some other target sets that have been previously used in the literature:

* *Gender target sets taken from Nosek, Banaji, and Greenwald 2002.*
    - Female: `sister, female, woman, girl, daughter, she, hers, her`.
    - Male: `brother, male, man, boy, son, he, his, him`.
* *Religion target sets taken from Garg et al. 2018.*
    - Islam: `allah, ramadan, turban, emir, salaam, sunni, koran, imam, sultan, prophet, veil, ayatollah, shiite, mosque, islam, sheik, muslim, muhammad`.
    - Christianity: `baptism, messiah, catholicism, resurrection, christianity, salva-tion, protestant, gospel, trinity, jesus, christ, christian, cross,catholic, church`.
* *Racial target sets taken from Garg et al. 2017*
    - White last names: `harris, nelson, robinson, thompson, moore, wright, anderson, clark, jackson, taylor, scott, davis, allen, adams, lewis, williams, jones, wilson, martin, johnson`.
    - Hispanic last names: `ruiz, alvarez, vargas, castillo, gomez, soto,gonzalez, sanchez, rivera, mendoza, martinez, torres, ro-driguez, perez, lopez, medina, diaz, garcia, castro, cruz`.
    - Asian last names: `cho, wong, tang, huang, chu, chung, ng,wu, liu, chen, lin, yang, kim, chang, shah, wang, li, khan,singh, hong`.
    - Russian last names: `gurin, minsky, sokolov, markov, maslow, novikoff, mishkin, smirnov, orloff, ivanov, sokoloff, davidoff, savin, romanoff, babinski, sorokin, levin, pavlov, rodin, agin`.
* *Career/family target sets taken from Garg et al. 2018.*
    - Career: `executive, management, professional, corporation, salary, office, business, career`.
    - Family: `home, parents, children, family, cousins, marriage, wedding, relatives.Math: math, algebra, geometry, calculus, equations, computation, numbers, addition`.
* *Arts/Science target sets taken from Garg et al. 2018.*
    - Arts: `poetry, art, sculpture, dance, literature, novel, symphony, drama`.
    - Science: `science, technology, physics, chemistry, Einstein, NASA, experiment, astronomy`.